# 模块 10 — 三个 Marangoni 扩展的对比演示

基线是**仅热传导**的 Goldak 三维温度场 (模块 4)。三个 Marangoni 扩展在其上
以**递增的保真度**加入热毛细 (Marangoni) 对流搅拌:

| 扩展 | 维度 / 方法 | 求解内容 | 计算成本 |
|---|---|---|---|
| **10A** `EffectiveMarangoniCorrection` | 0 维参数化 | 由 Pe 数把池内热扩散率折算为 `α_eff`, 再修正池形 | 闭式, 瞬时 |
| **10B** `SurfaceMarangoniFlow2D` | 2D 顶面 (x-y) | 由表面切应力 `τ=dγ/dT·∇T` 给出表面回流速度场 | N 步显式 |
| **10C** `IncompressibleMarangoniFlow2D` | 2D 纵截面 (x-z) | 流函数-涡量不可压流, 自由面剪切驱动环流并对流输运热量 | N 步 × 泊松迭代 |

> **说明**: 三者都是*降阶研究原型* (带保守限速/限幅), 用于和仅传导场作定性对比,
> 而非标定的生产级 CFD。所给速度/Pe 为量级估计。

本 notebook 全部用 **matplotlib** 内联绘图 (无需 trame/pyvista), 可直接逐格运行。

```bash
uv run jupyter lab notebooks/marangoni_comparison.ipynb
```

## 0. 基线: 仅传导的 Goldak 熔池

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from welding_dynamics import (GoldakFDM, EffectiveMarangoniCorrection,
                              SurfaceMarangoniFlow2D, IncompressibleMarangoniFlow2D)

# 基线: 仅传导的 Goldak 三维温度场 (模块 4)
g = GoldakFDM()
g.run(t_end=5.0)
dx = g.dx
L, W, D = g.pool_size()
print(f"conduction-only Goldak pool:  L x W x D = {L:.1f} x {W:.1f} x {D:.1f} mm")
print(f"peak temperature: {g.peak.max():.0f} K,  grid {g.Nx}x{g.Ny}x{g.Nz} (half model)")

## 1. 模块 10A — 有效热扩散率修正

把热毛细搅拌折算成放大的池内热扩散率 `α_eff`。下面 (左) 展示有效扩散倍率随表面张力梯度幅值上升并在保守限幅器处饱和; (右) 外向流使熔池**变宽变浅**, 内向流使其**变窄变深** —— 这正是 Marangoni 流改变焊透形貌的经典机理。

In [ ]:
# 模块 10A: 把热毛细搅拌折算成"有效热扩散率" (0 维闭式修正)
eff = EffectiveMarangoniCorrection()
a_strength = eff.strength()        # Ma, Pe, u_surface, direction
a_mult = eff.alpha_eff() / eff.alpha
print(f"Ma={a_strength['Ma']:.0f}  Pe={a_strength['Pe']:.0f}  "
      f"u_surface~{abs(a_strength['u_surface']):.2f} m/s  flow={a_strength['direction']}")
print(f"effective diffusivity  alpha_eff/alpha = {a_mult:.2f}  (limiter = {eff.limiter})")

# (a) 有效扩散倍率随表面张力梯度幅值变化 (展示保守限幅器的饱和)
mags = np.linspace(0.2e-4, 6e-4, 25)
mult_curve = [EffectiveMarangoniCorrection(dgamma_dT=-m).alpha_eff() / eff.alpha
              for m in mags]

# (b) 池形重塑: 外向流(洁净钢, dgamma/dT<0) vs 内向流(活性元素, dgamma/dT>0)
out = EffectiveMarangoniCorrection(dgamma_dT=-4e-4).corrected_pool_size(L, W, D)
inw = EffectiveMarangoniCorrection(dgamma_dT=+4e-4).corrected_pool_size(L, W, D)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(mags * 1e4, mult_curve, "o-")
ax1.axhline(eff.limiter, ls="--", color="r", label=f"limiter = {eff.limiter}")
ax1.set_xlabel(r"|d$\gamma$/dT|  [10$^{-4}$ N/(m K)]")
ax1.set_ylabel(r"$\alpha_{eff}/\alpha$")
ax1.set_title("10A: effective stirring saturates at limiter")
ax1.legend(); ax1.grid(alpha=0.3)

xb = np.arange(3); bw = 0.27
ax2.bar(xb - bw, [L, W, D], bw, label="conduction", color="gray")
ax2.bar(xb,      out,        bw, label=r"outward (d$\gamma$/dT<0)", color="tab:blue")
ax2.bar(xb + bw, inw,        bw, label=r"inward (d$\gamma$/dT>0)", color="tab:orange")
ax2.set_xticks(xb); ax2.set_xticklabels(["L", "W", "D"]); ax2.set_ylabel("mm")
ax2.set_title("10A: predicted pool reshaping")
ax2.legend(); ax2.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print(f"outward L x W x D = {out[0]:.1f} x {out[1]:.1f} x {out[2]:.1f} mm  (wider, shallower)")
print(f"inward  L x W x D = {inw[0]:.1f} x {inw[1]:.1f} x {inw[2]:.1f} mm  (narrower, deeper)")

## 2. 模块 10B — 2D 表面热毛细回流

在熔池**顶面**上由表面温度梯度计算热毛细切应力驱动的回流速度场 (流函数投影保持面内无散)。箭头为瞬时速度, 颜色为速度大小; 青色为熔合线。

In [ ]:
# 模块 10B: 顶面 (x-y) 热毛细回流速度场。取 Goldak 峰值场顶层, 沿 y=0 镜像为全宽
Ttop = g.peak[:, :, 0]
Tfull = np.concatenate([Ttop[:, :0:-1], Ttop], axis=1)
x = g.x * 1e3
yfull = np.concatenate([-g.y[:0:-1], g.y]) * 1e3
Xg, Yg = np.meshgrid(x, yfull, indexing="ij")
mask = Tfull >= g.Tm

surf = SurfaceMarangoniFlow2D()
b_u, b_v = surf.velocity(Tfull, dx, dx, melt_mask=mask)
b_diag = surf.diagnostics(Tfull, dx, dx, melt_mask=mask)
print(f"surface flow: max {b_diag['max_speed']*1e3:.0f} mm/s, "
      f"mean(in pool) {b_diag['mean_speed']*1e3:.0f} mm/s, "
      f"Pe~{b_diag['Pe']:.0f}, {b_diag['direction']}")
print("(reduced kinematic model: central-difference advection is not meant for "
      "long-time stepping; we visualize the instantaneous velocity field.)")

fig, ax = plt.subplots(figsize=(9, 4.2))
pc = ax.pcolormesh(Xg, Yg, Tfull, cmap="inferno", shading="auto")
ax.contour(Xg, Yg, Tfull, levels=[g.Tm], colors="cyan", linewidths=1.5)
spd = np.hypot(b_u, b_v)
s = 2
ax.quiver(Xg[::s, ::s], Yg[::s, ::s], b_u[::s, ::s], b_v[::s, ::s],
          spd[::s, ::s], cmap="cool", width=0.004)
ii, jj = np.where(mask)
ax.set_xlim(x[ii.min()] - 4, x[ii.max()] + 4)
ax.set_ylim(yfull[jj.min()] - 4, yfull[jj.max()] + 4)
ax.set_xlabel("x along weld [mm]"); ax.set_ylabel("y across weld [mm]")
ax.set_title("10B: thermocapillary surface velocity (cyan = melt line)")
ax.set_aspect("equal"); fig.colorbar(pc, ax=ax, label="T [K]")
plt.tight_layout(); plt.show()

## 3. 模块 10C — 不可压纵截面环流

在 **x-z 纵截面**上求解流函数-涡量不可压流: 自由面热毛细剪切注入涡量, 形成熔池环流胞, 并用速度场对流输运温度。左图为流函数 ψ 与速度矢量, 右图为对流后的温度场。

In [ ]:
# 模块 10C: 纵向截面 (x-z, y=0 对称面) 流函数-涡量不可压环流, 时间推进至准稳态
Txz = g.peak[:, 0, :].copy()
x = g.x * 1e3
z = g.z * 1e3
Xg, Zg = np.meshgrid(x, z, indexing="ij")
mask = Txz >= g.Tm

inc = IncompressibleMarangoniFlow2D()
c_state = inc.initial_state(Txz.shape, T0=g.T0)
c_state["T"] = Txz.copy()
dt = inc.stable_dt(dx, dx, max_speed=inc.speed_limiter)
for _ in range(120):
    c_state = inc.step(c_state, dx, dx, dt, melt_mask=mask)
c_diag = inc.diagnostics(c_state, dx, dx, melt_mask=mask)
print(f"recirculation: max {c_diag['max_speed']*1e3:.0f} mm/s, "
      f"mean {c_diag['mean_speed']*1e3:.0f} mm/s, Pe~{c_diag['Pe']:.0f}, {c_diag['direction']}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
cf = ax1.contourf(Xg, Zg, c_state["psi"], levels=18, cmap="RdBu_r")
sp = 2
ax1.quiver(Xg[::sp, ::sp], Zg[::sp, ::sp],
           c_state["u"][::sp, ::sp], c_state["w"][::sp, ::sp], color="k")
ax1.contour(Xg, Zg, Txz, levels=[g.Tm], colors="lime", linewidths=1.5)
ax1.invert_yaxis(); ax1.set_xlabel("x [mm]"); ax1.set_ylabel("depth z [mm]")
ax1.set_title(r"10C: streamfunction $\psi$ + velocity (lime = melt line)")
fig.colorbar(cf, ax=ax1, label=r"$\psi$")

pc = ax2.pcolormesh(Xg, Zg, c_state["T"], cmap="inferno", shading="auto")
ax2.contour(Xg, Zg, Txz, levels=[g.Tm], colors="cyan", linewidths=1.0)
ax2.invert_yaxis(); ax2.set_xlabel("x [mm]"); ax2.set_ylabel("depth z [mm]")
ax2.set_title("10C: temperature with convection")
fig.colorbar(pc, ax=ax2, label="T [K]")

ii, kk = np.where(mask)
for ax in (ax1, ax2):
    ax.set_xlim(x[ii.min()] - 4, x[ii.max()] + 4)
plt.tight_layout(); plt.show()

## 4. 三者对比

把三个模型产生的特征速度尺度与 Péclet 数并排比较。注意各模型的 Pe 用了**不同的特征长度**, 故跨模型的 Pe 数只作量级参考, 不宜直接定量对照。

In [ ]:
# 三个扩展的定量对比 (注意: 均为降阶研究原型, 带保守限速/限幅, 非标定 CFD)
models = ["10A\neffective", "10B\nsurface", "10C\nsection"]
speed_mm = [abs(a_strength["u_surface"]) * 1e3, b_diag["max_speed"] * 1e3,
            c_diag["max_speed"] * 1e3]
pe = [a_strength["Pe"], b_diag["Pe"], c_diag["Pe"]]
colors = ["tab:gray", "tab:green", "tab:purple"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ax1.bar(models, speed_mm, color=colors)
ax1.set_ylabel("velocity scale [mm/s]"); ax1.set_title("characteristic flow speed")
ax2.bar(models, pe, color=colors)
ax2.set_ylabel("Peclet number"); ax2.set_title("Pe (model-specific length scales)")
for ax in (ax1, ax2):
    ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print(f"{'model':<26}{'speed [mm/s]':>14}{'Pe':>10}{'direction':>12}")
print("-" * 62)
print(f"{'10A effective-diffusivity':<26}{speed_mm[0]:>14.0f}{pe[0]:>10.0f}"
      f"{a_strength['direction']:>12}")
print(f"{'10B surface flow (x-y)':<26}{speed_mm[1]:>14.0f}{pe[1]:>10.0f}{b_diag['direction']:>12}")
print(f"{'10C incompressible (x-z)':<26}{speed_mm[2]:>14.0f}{pe[2]:>10.0f}{c_diag['direction']:>12}")

## 5. 小结

- **10A** 最便宜 (闭式), 适合快速判断 Marangoni 是否会显著改变池形 (变宽变浅 / 变窄变深), 但不给流场。
- **10B** 给出顶面回流**运动学**, 直观显示横向搅拌方向; 中心差分对流不适合长时间推进。
- **10C** 唯一真正解不可压**环流**并耦合对流换热, 最接近真实 Marangoni 熔池, 代价是每步一次泊松求解。
- 三者相对仅传导基线都预测**更强的横向/环流热输运**, 倾向于把热量从池心带向边缘 (洁净钢 dγ/dT<0 的外向流), 与实测 Marangoni 主导焊缝变宽变浅一致。